# Neural Fortran Examples: Copying Network Parameters

Created with Grok Build

Source code by [Milan Curcic](https://github.com/milancurcic) and the neural-fortran contributors, from the [neural-fortran](https://github.com/modern-fortran/neural-fortran) repository. The library is released under the MIT license. Copyright (c) 2018-2025 neural-fortran contributors. The paper is Milan Curcic, [A parallel Fortran framework for neural networks and deep learning](https://doi.org/10.1145/3323057.3323059), *ACM SIGPLAN Fortran Forum* 38(1), 2019.

## Introduction

This is `example/get_set_network_params.f90`. It trains one small sine network, reads every weight and bias with `get_params`, writes that vector into a second network with `set_params`, and checks that the two networks agree on a fixed test grid.

## The sine target

```{literalinclude} Fortran_Code/Section_Examples_Get_Set/app/get_set_network_params.f90
:language: fortran
:lines: 21-48
```

`net1` is `input(1)`, `dense(5)`, `dense(1)`. Both dense layers use the default sigmoid. The parameter count is $(1 \cdot 5 + 5) + (5 \cdot 1 + 1) = 16$, which `get_num_params` prints after training.

The test grid is fixed. With `test_size = 30`,

$$
x_i = (i-1)\,\frac{2\pi}{30}, \qquad y_i = \frac{\sin x_i + 1}{2}, \qquad i = 1,\ldots,30.
$$

The shift and scale put the sine into $(0, 1)$, which is the range of a sigmoid output. Each training step draws a fresh $x$ from $[0, 2\pi)$ and sets $y$ from the same formula. There is no stored training set. The update is SGD with learning rate 1, once per random sample, for 100001 steps because `n` runs from 0 through 100000.

The number the loop prints is not the library's quadratic loss. It is the mean squared error on the 30 test points:

$$
\mathrm{MSE} = \frac{1}{N}\sum_{i=1}^{N}(\hat y_i - y_i)^2.
$$


## Copying the vector

```{literalinclude} Fortran_Code/Section_Examples_Get_Set/app/get_set_network_params.f90
:language: fortran
:lines: 50-90
```

`get_params` packs every weight and bias into one rank-1 array. `set_params` writes a vector of that length into a network of the same shape. `net2` is constructed with its own random weights, and the next line replaces them:

```fortran
call net2 % set_params(net1 % get_params())
```

The program then prints both vectors and both test-grid predictions. The last line is the Fortran reduction `all(ypred1 == ypred2)`. On this run it printed `T`. The comparison is exact equality of the default `real` values, which holds here because `net2` evaluated the same arithmetic on the same parameters. It is not a tolerance test.


## Program

```{literalinclude} Fortran_Code/Section_Examples_Get_Set/app/get_set_network_params.f90
:language: fortran
```

The package file below builds this source with fpm. The Fortran program itself is the file from the repository.

```{literalinclude} Fortran_Code/Section_Examples_Get_Set/fpm.toml
:language: toml
```

## Build

The program above was compiled from the neural-fortran repository at commit `357e63d` (`Support of the argument stride for locally_connected2d_layer`) with fpm 0.10.1:

```text
fpm build --profile release
```

The compiler was gfortran 14.2.0. The log is that release build. It compiles the library and every program in `example/`, including this one. The `fpm.toml` on this page is a standalone package for the copied source. It takes neural-fortran as a git dependency, in the same form as the sine-wave page.


```{literalinclude} Fortran_Code/fpm_release_build.log
:language: text
```

## Run

The binary was `get_set_network_params` from the release build. Standard error was empty and the process exited 0. The loss and the parameter vector depend on the random draws, so they belong to this run.

```{literalinclude} Fortran_Code/Section_Examples_Get_Set/get_set_network_params.out
:language: text
```

## What the run did

![Sine fit and the copied network](images/nf_get_set.png)

The test MSE starts at 0.132005, drops to 0.006663 by iteration 30000, and ends at 0.006520. It is not monotone: iteration 50000 is 0.007404. Five sigmoid units cannot match every wiggle of a sine, and each step is one random sample, so the test number moves after it is already small.

`get_num_params()` printed 16. The sixteen printed values are the same on `net1` and `net2`. The thirty test outputs match digit for digit, and the final line is `T`. The right-hand plot is `net1` on the test grid against $(\sin x + 1)/2$. The copy is the same curve.


## Conclusion

`get_params` and `set_params` are the supported way to move a trained network. The weight walk on the previous page reads the same numbers from inside `dense_layer`. This page treats them as one vector and shows that a second network with those values reproduces the outputs.